# Adaptive Nested-Error Nonparametric SAE — Walkthrough

This notebook runs the full pipeline: BHF vs. PM model fitting, the λ=0 test, and district-level
poverty indicators.

**Before running with real data:** point `DATA_DIR` at your local extraction of the NISR EICV7
`Cross_Section` folder (never commit that data to this repo — see `README.md`).

**To try it without real data first:** set `USE_SYNTHETIC = True` below and run everything —
this is a good way to confirm your environment is set up correctly.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root, if this notebook lives in notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.step2_models import fit_bhf, fit_pm, lambda_test
from src.step3_poverty_indicators import district_level_indicators
from src.step4_run_analysis import prepare_covariates, COVARIATES

print("Imports OK")
print("Covariates used:", COVARIATES)

## 2. Load data

Set `USE_SYNTHETIC = False` and `DATA_DIR` once you have real EICV7 data extracted locally.

In [ ]:
USE_SYNTHETIC = True
DATA_DIR = "/path/to/Cross_Section"  # only used if USE_SYNTHETIC = False

if USE_SYNTHETIC:
    from tests.generate_synthetic_data import generate
    df_raw = generate(n_districts=30, n_per_district=500)
    welfare_col = "sol_jan"
else:
    from src.step1_build_dataset import build_dataset
    df_raw = build_dataset(DATA_DIR)
    welfare_col = "sol_jan"  # IMPORTANT: not cons1ae -- see README for why

print(df_raw.shape)
df_raw.head()

## 3. Prepare covariates

In [ ]:
df = df_raw.rename(columns={welfare_col: "welfare"})
df = prepare_covariates(df)
print(f"Analysis sample: {len(df)} households, {df['district'].nunique()} districts")
df[COVARIATES + ["welfare"]].describe()

## 4. Fit BHF (standard model)

In [ ]:
bhf_fit = fit_bhf(df, "welfare", COVARIATES, "district")
print(bhf_fit.summary())

## 5. Fit PM (adaptive nonparametric model)

In [ ]:
pm_fit, knots, spline_cols = fit_pm(df, "welfare", COVARIATES, "district", "hhsize")
print(pm_fit.summary())

## 6. Test H0: λ = 0

Uses the correct boundary-adjusted null distribution (Self & Liang, 1987), not a naive chi-square
test -- see `docs/theory.pdf`, Proposition 3.

In [ ]:
sigma_g2 = pm_fit.vcomp[0] if len(pm_fit.vcomp) > 0 else 0.0
sigma_e2 = pm_fit.scale
lam = sigma_g2 / (sigma_g2 + sigma_e2)
lr_stat, p_val = lambda_test(bhf_fit, pm_fit)

print(f"lambda           = {lam:.5f}")
print(f"LR statistic     = {lr_stat:.4f}")
print(f"boundary-adjusted p-value = {p_val:.4f}")
print()
if p_val < 0.05:
    print("=> Reject H0: a statistically significant nonlinear departure from BHF was detected.")
else:
    print("=> Fail to reject H0: no significant evidence the nonlinear term is needed here.")

## 7. District-level poverty indicators

In [ ]:
POVERTY_LINE = 560127  # RWF per adult equivalent per year (EICV7) -- update if using a different survey

bhf_district = district_level_indicators(df, bhf_fit.fittedvalues.values, np.sqrt(bhf_fit.scale),
                                          "pop_wt", "district", POVERTY_LINE)
pm_district = district_level_indicators(df, pm_fit.fittedvalues.values, np.sqrt(pm_fit.scale),
                                         "pop_wt", "district", POVERTY_LINE)

comparison = bhf_district[["HCR", "PG"]].join(pm_district[["HCR", "PG"]], lsuffix="_BHF", rsuffix="_PM")
comparison["HCR_diff"] = comparison["HCR_PM"] - comparison["HCR_BHF"]
comparison.sort_values("HCR_diff")

## 8. National comparison

In [ ]:
national_bhf = np.average(bhf_district["HCR"], weights=bhf_district["n_households"])
national_pm = np.average(pm_district["HCR"], weights=pm_district["n_households"])
print(f"National HCR -- BHF: {national_bhf:.3f}%   PM: {national_pm:.3f}%")

## 9. Visualize: PM vs. BHF by district

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(comparison["HCR_BHF"], comparison["HCR_PM"])
lims = [comparison[["HCR_BHF", "HCR_PM"]].min().min(), comparison[["HCR_BHF", "HCR_PM"]].max().max()]
ax.plot(lims, lims, "--", color="gray")
ax.set_xlabel("BHF model-based HCR (%)")
ax.set_ylabel("PM model-based HCR (%)")
ax.set_title("District poverty headcount: PM vs BHF")
plt.tight_layout()
plt.show()

## Next steps

- If `USE_SYNTHETIC = False` worked with your real data: check `bhf_fit.summary()` and
  `pm_fit.summary()` for `Converged: Yes` before trusting any numbers.
- Review the covariate list in `src/step4_run_analysis.py` (`COVARIATES`) — the current set is
  minimal/illustrative.
- No bootstrap MSE / confidence intervals are computed yet — see `README.md` "Status" section.